# Jamii Afya Phase 0/1 — current training-data audit

This evaluation-only notebook rebuilds the public default training inputs and measures their effective token distribution. It does not train or modify a model.

In [ ]:
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
RESULTS = WORK / 'phase01-results'
BRANCH = 'research/phase-0-1-baseline-evals'

def run(command, cwd=None, log=None):
    print('+', ' '.join(command), flush=True)
    result = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, check=False)
    print(result.stdout, flush=True)
    if log:
        Path(log).parent.mkdir(parents=True, exist_ok=True)
        Path(log).write_text(result.stdout, encoding='utf-8')
    if result.returncode:
        raise RuntimeError(f'command failed with exit {result.returncode}: {command}')
    return result

RESULTS.mkdir(parents=True, exist_ok=True)
run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
     'https://github.com/qeinstein/adtc-llm-limited-hardware.git', str(REPO)])
run([sys.executable, '-m', 'pip', 'install', '-q',
     'datasets==4.8.5', 'transformers==5.16.1'])
print({'python': sys.version, 'platform': platform.platform()})

In [ ]:
run([sys.executable, 'scripts/build_accuracy_sft.py',
     '--max-per-dataset', '20000'], cwd=REPO,
    log=RESULTS / 'build_accuracy_sft.log')
run([sys.executable, 'scripts/build_healthcare_corpus.py',
     '--max-per-dataset', '15000'], cwd=REPO,
    log=RESULTS / 'build_healthcare_corpus.log')

In [ ]:
audit_dir = RESULTS / 'current_default'
run([sys.executable, 'scripts/audit_training_data.py',
     '--manifest', 'experiments/data/current_default.json',
     '--output-dir', str(audit_dir)], cwd=REPO,
    log=RESULTS / 'audit.log')

provenance = {
    'repository_sha': run(['git', 'rev-parse', 'HEAD'], cwd=REPO).stdout.strip(),
    'branch': BRANCH,
    'python': sys.version,
    'platform': platform.platform(),
    'datasets_version': __import__('datasets').__version__,
    'transformers_version': __import__('transformers').__version__,
    'tokenizer_revision': 'da87bfb608c14b7cf20ba1ce41287e8de496c0cd',
}
(RESULTS / 'run_provenance.json').write_text(
    json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
print((audit_dir / 'audit.md').read_text(encoding='utf-8'))